# ch12 练习 —— 窗口计算

> 数据：`data/load_curve.csv`（3 个台区 × 2160 小时，2026-01-01 ~ 2026-03-31）
> 单台区序列已备好：`load`（RangeIndex）、`load_ts`（DatetimeIndex）

**做题规则**：从上往下依次填，每个 `____` 处跑通再往下。
验收块 `assert` 不要改，它是你自己对账用的。

| 题号 | 考点 |
|---|---|
| TODO(1) | 固定窗口均值 + `min_periods` 默认值 |
| TODO(2) | `min_periods=1` 的代价与收益 |
| TODO(3) | `shift` / `diff` / `pct_change` 造滞后特征 |
| TODO(4) | `expanding` 累计均值 |
| TODO(5) | `ewm` 的 `span` 与 `adjust` |
| TODO(6) | 时间窗口 `rolling("24h")` |
| TODO(7) | 证明窗口按**行序**滑 |
| TODO(8) | 标签泄漏：`shift(1)` 打底 |
| TODO(9) | 分组滚动与两级 `MultiIndex` |
| TODO(10) | 分组 `shift` / `diff` 的索引差异 |
| TODO(11) | `rolling().apply` 自定义窗口统计 |
| TODO(12) | 动态阈值 vs 全局阈值（告警口径之争） |
| 综合① | 24h 时序特征表 + 泄漏自检 |
| 综合② | 分组滑动特征回贴原表 + 对账 |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

KEY_COL = "台区编号"
VALUE_COL = "负荷值"
STATION = "STATION_A_01"

# 负荷曲线：3 个台区 x 2160 小时（2026-01-01 ~ 2026-03-31 逐小时）
curve = pd.read_csv(DATA / "load_curve.csv", parse_dates=["时间戳"])
curve = curve.sort_values([KEY_COL, "时间戳"]).reset_index(drop=True)

# 单台区 + DatetimeIndex：时间窗口 rolling("24h") 必须用它
station = curve[curve[KEY_COL] == STATION].set_index("时间戳")
load_ts = station[VALUE_COL]

# 单台区 + RangeIndex：固定窗口 rolling(24) 用这个
load = load_ts.reset_index(drop=True)

# 分组对象（跨台区）
gb = curve.groupby(KEY_COL)

print("pandas", pd.__version__)
print("curve", curve.shape, "| 单台区", load.shape, "| 台区数", curve[KEY_COL].nunique())

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def window_span(w):
    """窗口极差（max - min），给 rolling().apply 用。"""
    return float(np.nanmax(w) - np.nanmin(w))


def peek(series, n=3):
    """看一下前 n 个非缺失值，省得每次手打 dropna().head()。"""
    return series.dropna().head(n).round(4).tolist()


print("load 缺失:", int(load.isna().sum()), "| load_ts 索引:", type(load_ts.index).__name__)

---

## 基础题

In [ ]:
roll_default = load.rolling(24).mean()
n_nan = int(roll_default.isna().sum())
# ---- 验收 ----
assert n_nan == 319, f"默认参数下应有 319 个 NaN，实际 {n_nan}"
assert abs(roll_default.iloc[23] - load.iloc[:24].mean()) < 1e-9, "roll[23] 应等于前 24 行均值"
assert bool(pd.isna(roll_default.iloc[22])), "第 22 行窗口未满，应为 NaN"
print("TODO(1) 通过 -> NaN", n_nan, "| roll[23] =", round(roll_default.iloc[23], 4))
print("   注意：NaN 数远大于「头部 23 + 序列缺失 23」，缺失会污染后续窗口")

In [ ]:
#         并把第 0 行的值存入 first_mp1
roll_mp1 = load.rolling(24, min_periods=1).mean()
n_nan_mp1 = int(roll_mp1.isna().sum())
first_mp1 = float(roll_mp1.iloc[0])
# ---- 验收 ----
assert n_nan_mp1 == 0, f"min_periods=1 时不该有 NaN，实际 {n_nan_mp1}"
assert abs(first_mp1 - load.iloc[0]) < 1e-9, "第 0 行窗口只有 1 个点，应等于原值本身"
assert int(load.isna().sum()) == 23, "单台区缺失应为 23"
assert abs(roll_mp1.iloc[23] - roll_default.iloc[23]) < 1e-9, "第 23 行两版应一致（窗口都满 24）"
print("TODO(2) 通过 -> NaN 0 | first =", first_mp1, "| iloc[23] 两版一致")

In [ ]:
lag1 = load.shift(1)
lag24 = load.shift(24)
pc24 = load.pct_change(24, fill_method=None)
# ---- 验收 ----
assert int(lag1.isna().sum()) == 24, f"滞后1h 应有 24 个 NaN（首行 1 + 原缺失 23），实际 {int(lag1.isna().sum())}"
assert lag24.iloc[24] == load.iloc[0], "lag24[24] 应等于 load[0]"
assert abs(pc24.iloc[24] - (load.iloc[24] - load.iloc[0]) / load.iloc[0]) < 1e-9, "日环比公式应成立"
assert bool(np.allclose(load.diff(1).dropna(), (load - lag1).dropna())), "diff(1) 应等于 s - s.shift(1)"
print("TODO(3) 通过 -> lag1 NaN", int(lag1.isna().sum()), "| lag24[24] =", round(lag24.iloc[24], 2),
      "| 日环比[24] =", round(pc24.iloc[24], 4))

In [ ]:
exp_mean = load.expanding().mean()
n_nan_exp = int(exp_mean.isna().sum())
# ---- 验收 ----
assert n_nan_exp == 0, "expanding 默认 min_periods=1，不该有 NaN"
assert abs(exp_mean.iloc[0] - load.iloc[0]) < 1e-9, "第 0 行累计均值 = 原值"
assert abs(exp_mean.iloc[-1] - load.mean()) < 1e-6, "最后一行累计均值 = 全序列均值"
assert int(load.expanding(min_periods=24).mean().isna().sum()) == 23, "min_periods=24 时应有 23 个 NaN"
print("TODO(4) 通过 -> 累计末值", round(exp_mean.iloc[-1], 4), "| 序列均值", round(load.mean(), 4))

In [ ]:
#         布尔结果存入 same；再算 adjust=False 版本存入 ewm_noadj
ewm_span = load.ewm(span=24).mean()
same = bool(np.allclose(ewm_span, load.ewm(alpha=2 / 25).mean()))
ewm_noadj = load.ewm(span=24, adjust=False).mean()
# ---- 验收 ----
assert same is True, "span=24 应等价于 alpha=2/25"
assert abs(ewm_span.iloc[0] - load.iloc[0]) < 1e-9, "ewm 首值应等于原序列首值"
alpha = 2 / 25
manual = (1 - alpha) * load.iloc[0] + alpha * load.iloc[1]
assert abs(ewm_noadj.iloc[1] - manual) < 1e-9, "adjust=False 应符合 y=(1-a)y+ax 递推"
assert abs(ewm_span.iloc[-1] - ewm_noadj.iloc[-1]) < 1.0, "两者末期应已接近收敛"
print("TODO(5) 通过 -> span==alpha:", same, "| adjust=False y[1] =", round(ewm_noadj.iloc[1], 4))

In [ ]:
#         再用 min_periods=24 算一份存入 time_roll_mp，两个 NaN 数分别存入 n_time / n_time_mp
time_roll = load_ts.rolling("24h").mean()
time_roll_mp = load_ts.rolling("24h", min_periods=24).mean()
n_time = int(time_roll.isna().sum())
n_time_mp = int(time_roll_mp.isna().sum())
# ---- 验收 ----
assert n_time == 0, f"时间窗口默认 min_periods=1，不该有 NaN，实际 {n_time}"
assert n_time_mp == 319, f"min_periods=24 时应与固定窗口一致（319），实际 {n_time_mp}"
assert abs(time_roll.iloc[23] - roll_default.iloc[23]) < 1e-9, "等间隔数据下时间窗口与固定窗口应一致"
assert abs(time_roll.iloc[0] - load_ts.iloc[0]) < 1e-9, "第 0 行窗口只有 1 个点"
# RangeIndex 上应当直接报错
try:
    load.rolling("24h").mean()
    raise AssertionError("RangeIndex 上用时间窗口应该抛 ValueError")
except AssertionError:
    raise
except ValueError:
    pass
print("TODO(6) 通过 -> 默认", n_time, "个 NaN | min_periods=24 时", n_time_mp, "个")

---

## 进阶题

In [ ]:
#         算它的 24 窗口均值存入 rev_roll；验证 rev_roll[23] 等于原序列最后 24 行的均值
rev = load.iloc[::-1].reset_index(drop=True)
rev_roll = rev.rolling(24).mean()
# ---- 验收 ----
assert abs(rev_roll.iloc[23] - load.iloc[-24:].mean()) < 1e-9, "倒序后第 23 行窗口 = 原序列末 24 行"
assert abs(rev_roll.iloc[23] - roll_default.iloc[23]) > 1, "倒序结果必须与原序不同（否则说明不是按行序）"
assert pd.isna(rev_roll.iloc[22]), "倒序后第 22 行窗口仍未满"
print("TODO(7) 通过 -> 倒序 roll[23] =", round(rev_roll.iloc[23], 4),
      "| 原序 roll[23] =", round(roll_default.iloc[23], 4), "→ 完全不同，证明只认行序")

In [ ]:
#         两者的 NaN 个数分别存入 n_leak / n_safe
leak = load.rolling(24).mean()
safe = load.shift(1).rolling(24).mean()
n_leak = int(leak.isna().sum())
n_safe = int(safe.isna().sum())
# ---- 验收 ----
assert abs(leak.iloc[23] - load.iloc[:24].mean()) < 1e-9, "泄漏版 roll[23] 含 load[23] 自己"
assert abs(safe.iloc[24] - load.iloc[:24].mean()) < 1e-9, "安全版第 24 行才等于前 24 行均值"
assert pd.isna(safe.iloc[23]), "安全版第 23 行窗口不足，应为 NaN"
assert n_safe == n_leak + 1, f"安全版应恰好多 1 个 NaN（{n_leak+1}），实际 {n_safe}"
assert bool(np.allclose((leak - safe).dropna().abs().gt(0).sum() > 0, True)), "两版数值必须有差异"
print("TODO(8) 通过 -> 泄漏版", n_leak, "个 NaN | 安全版", n_safe, "个（多 1 个是正确代价）")

In [ ]:
#         各组「首个有效值」拿不到就遍历，先用 to_frame 存一份 gb_roll_df（列名改为「窗口均值」）
gb_roll = gb[VALUE_COL].rolling(24).mean()
n_gb_roll = int(gb_roll.isna().sum())
gb_roll_df = gb_roll.rename(VALUE_COL).reset_index()
# ---- 验收 ----
assert n_gb_roll == 829, f"分组滚动应有 829 个 NaN（3 组 × 含组内头部与缺失污染），实际 {n_gb_roll}"
assert gb_roll.index.names == [KEY_COL, None], f"应返回两级 MultiIndex，实际 {gb_roll.index.names}"
assert list(gb_roll_df.columns) == [KEY_COL, "level_1", VALUE_COL], \
    f"reset_index 后列名应为 [台区编号, level_1, 负荷值]，实际 {list(gb_roll_df.columns)}"
assert gb_roll_df.shape == (6480, 3), f"形状应为 (6480, 3)，实际 {gb_roll_df.shape}"
assert int(gb_roll_df[VALUE_COL].notna().sum()) == 6480 - 829, "有效值个数应守恒"
print("TODO(9) 通过 -> 索引层级", gb_roll.index.names, "| NaN", n_gb_roll,
      "| reset 后列名", list(gb_roll_df.columns))

In [ ]:
#          gb_shift = 组内滞后 1，gb_diff = 组内一阶差分；
#          两者 NaN 数分别存入 n_gb_shift / n_gb_diff
gb_shift = gb[VALUE_COL].shift(1)
gb_diff = gb[VALUE_COL].diff(1)
n_gb_shift = int(gb_shift.isna().sum())
n_gb_diff = int(gb_diff.isna().sum())
# ---- 验收 ----
assert gb_shift.index.names == [None], f"分组 shift 应返回原索引（单层），实际 {gb_shift.index.names}"
assert gb_diff.index.names == [None], "分组 diff 也应返回原索引"
assert len(gb_shift) == 6480, "长度应与原表一致，可直接回贴"
assert n_gb_shift == 51, f"分组 shift NaN 应为 51（每组首行 3 个 + 原有缺失 48），实际 {n_gb_shift}"
assert n_gb_diff == 84, f"分组 diff NaN 应为 84，实际 {n_gb_diff}"
# 组边界：每组第一行必为 NaN
first_rows = gb_shift.groupby(curve[KEY_COL]).head(1)
assert bool(first_rows.isna().all()), "每组第一行必须是 NaN（组内无前一行）"
print("TODO(10) 通过 -> shift NaN", n_gb_shift, "| diff NaN", n_gb_diff,
      "| 索引层级", gb_shift.index.names, "（与 rolling 的两级索引不同）")

In [ ]:
#           （窗口函数用脚手架里的 window_span，raw=True）
roll_span = load.rolling(24, min_periods=24).apply(window_span, raw=True)
# ---- 验收 ----
assert int(roll_span.isna().sum()) == 319, f"与 rolling(24).max() 的 NaN 模式应一致（319），实际 {int(roll_span.isna().sum())}"
manual_first = load.iloc[:24].max() - load.iloc[:24].min()
assert abs(roll_span.iloc[23] - manual_first) < 1e-9, "第 23 行应等于前 24 行的 max-min"
# 与 max()-min() 两种写法对账
pair = load.rolling(24).max() - load.rolling(24).min()
assert bool(np.allclose(roll_span.dropna(), pair.dropna())), "apply 写法应等价于 max()-min()"
print("TODO(11) 通过 -> roll_span[23] =", round(roll_span.iloc[23], 4),
      "| 与 max-min 完全一致:", True)

In [ ]:
#           全局 P95 存入 thr_glob；滑动 P95（24 窗口、min_periods=12）存入 thr_dyn
#           两个布尔标记存入 flag_glob / flag_dyn（load 是否超过各自阈值）
#        thr_dyn = load.rolling(24, min_periods=12).quantile(0.95)
#        flag_glob = load > thr_glob ；flag_dyn = load > thr_dyn
thr_glob = load.quantile(0.95)
thr_dyn = load.rolling(24, min_periods=12).quantile(0.95)
flag_glob = load > thr_glob
flag_dyn = load > thr_dyn
# ---- 验收 ----
assert abs(float(thr_glob) - 1025.496) < 1e-2, f"全局 P95 应为 1025.50，实际 {float(thr_glob):.3f}"
assert int(thr_dyn.isna().sum()) == 11, f"滑动 P95 应有 11 个 NaN，实际 {int(thr_dyn.isna().sum())}"
assert float(thr_dyn.mean()) < float(thr_glob), "滑动阈值均值应低于全局阈值（局部口径更敏感）"
assert int(flag_glob.sum()) == 107, f"全局口径应判出 107 条，实际 {int(flag_glob.sum())}"
assert int(flag_dyn.sum()) == 194, f"滑动口径应判出 194 条，实际 {int(flag_dyn.sum())}"
assert int((flag_glob & flag_dyn).sum()) == 55, f"重叠应为 55，实际 {int((flag_glob & flag_dyn).sum())}"
assert int((flag_dyn & ~flag_glob).sum()) == 139, f"全局漏掉的应有 139 条，实际 {int((flag_dyn & ~flag_glob).sum())}"
assert int((flag_glob & ~flag_dyn).sum()) == 52, f"滑动漏掉的应有 52 条，实际 {int((flag_glob & ~flag_dyn).sum())}"
# 被全局漏掉的都是「局部尖峰」：值不算全月最高，但相对本时段很高
missed = load[flag_dyn & ~flag_glob]
assert float(missed.min()) > 600, f"被全局漏掉的值应偏小，实际最小值 {float(missed.min()):.2f}"
assert float(missed.max()) <= float(thr_glob), "被全局漏掉的必然都 ≤ 全局阈值"
# 附：center=True 的平滑边界
assert int(load.rolling(24, center=True, min_periods=12).mean().isna().sum()) == 0, \
    "center=True + min_periods=12 时不应有 NaN"
assert int(load.rolling(24, center=True, min_periods=24).mean().isna().sum()) == 319, \
    "center=True + min_periods=24 的 NaN 数与普通 rolling(24) 相同"
print("TODO(12) 通过 -> 全局判", int(flag_glob.sum()), "条 | 滑动判", int(flag_dyn.sum()), "条",
      f"| 全局漏 {int((flag_dyn & ~flag_glob).sum())} 条局部尖峰")

---

## 综合题

### 综合 ① 24h 时序特征表（含泄漏自检）

造一张 6 特征的表，要求**全部特征列都由 `shift` 打底**，最后核对样本量。

In [ ]:
#        日环比用 load.pct_change(24, fill_method=None)；累计均值 load.expanding().mean()
feat = pd.DataFrame(index=load.index)
feat["负荷值"] = load
feat["滑动均值24"] = load.shift(1).rolling(24).mean()
feat["滑动标准差24"] = load.shift(1).rolling(24).std()
feat["滞后1h"] = load.shift(1)
feat["滞后24h"] = load.shift(24)
feat["日环比"] = load.pct_change(24, fill_method=None)
feat["累计均值"] = load.expanding().mean()
# ---- 验收 ----
assert feat.shape == (2160, 7), f"形状应为 (2160, 7)，实际 {feat.shape}"
assert list(feat.columns) == [
    "负荷值", "滑动均值24", "滑动标准差24", "滞后1h", "滞后24h", "日环比", "累计均值"
], f"列名不对：{list(feat.columns)}"
# 自检 1：滑动列比「直接用 rolling」多 1 个 NaN（安全版代价）
direct = load.rolling(24).mean()
assert int(feat["滑动均值24"].isna().sum()) == int(direct.isna().sum()) + 1, "shift 打底应恰好多 1 个 NaN"
# 自检 2：泄漏检查 —— 滑动均值必须等于「前 24 小时」，而不是「含当前行的 24 行」
i = len(feat) - 1
window_safe = load.iloc[i - 24:i].mean()     # 过去 24 行（不含第 i 行）
window_leak = load.iloc[i - 23:i + 1].mean()  # 含第 i 行
assert abs(feat["滑动均值24"].iloc[i] - window_safe) < 1e-9, "特征列应等于不含当前行的窗口均值"
assert abs(feat["滑动均值24"].iloc[i] - window_leak) > 1e-6, "特征列不得等于含当前行的窗口均值"
# 自检 3：滞后 24h 的自相关应显著高于滞后 1h（日周期）
corr24 = load.corr(feat["滞后24h"])
corr1 = load.corr(feat["滞后1h"])
assert corr24 > corr1, f"日周期应使 lag24 相关性更高：{corr24:.4f} vs {corr1:.4f}"
# 自检 4：样本量守恒
n_complete = len(feat.dropna())
assert n_complete == 1829, f"dropna 后应剩 1829 行，实际 {n_complete}"
assert n_complete / len(feat) > 0.8, "完整样本占比应 > 80%"
print("综合① 通过 ->", feat.shape, "| dropna 后", n_complete, "行")
print(f"   日周期自相关 {corr24:.4f} > 小时自相关 {corr1:.4f}")
print(feat.iloc[-1].round(4).to_string())

### 综合 ② 分组滑动特征回贴原表

把三件事一次做完，并证明"回贴对得上"：

1. 组内 24h 滑动均值、组内 24h 滞后值
2. 回贴到 `curve` 原表，**不能出现整列 NaN**
3. 对账：回贴后的值必须与手工按组计算的窗口值一致

In [ ]:
#         并把「整表行数 / 三列各自的 NaN 数」存入 report 字典
gb_roll24 = gb[VALUE_COL].transform(lambda s: s.rolling(24).mean())
curve2 = curve.copy()
curve2["组内滑动均值"] = gb_roll24
curve2["组内滞后24"] = gb[VALUE_COL].shift(24)
curve2["组内日环比"] = gb[VALUE_COL].transform(lambda s: s.pct_change(24, fill_method=None))
report = {
    "行数": len(curve2),
    "滑动均值NaN": int(curve2["组内滑动均值"].isna().sum()),
    "滞后24NaN": int(curve2["组内滞后24"].isna().sum()),
    "日环比NaN": int(curve2["组内日环比"].isna().sum()),
}
# ---- 验收 ----
assert set(report) == {"行数", "滑动均值NaN", "滞后24NaN", "日环比NaN"}
assert report["行数"] == 6480, f"行数应为 6480，实际 {report['行数']}"
assert report["滑动均值NaN"] == 829, f"组内滑动均值应有 829 个 NaN，实际 {report['滑动均值NaN']}"
assert report["滞后24NaN"] == 120, \
    f"组内滞后24 应有 120 个 NaN（每组前 24 行 72 个 + 原缺失 48 个），实际 {report['滞后24NaN']}"
assert report["日环比NaN"] == 168, f"组内日环比应有 168 个 NaN，实际 {report['日环比NaN']}"
# 关键自检：没有任何一列是「整列 NaN」（索引没对齐的信号）
assert all(curve2[c].notna().any() for c in ["组内滑动均值", "组内滞后24", "组内日环比"]), \
    "出现整列 NaN 说明索引没对齐"
# 对账：抽一个台区的第 30 行，手算组内前 24 小时均值
probe_rows = curve2[curve2[KEY_COL] == "STATION_C_01"].reset_index(drop=True)
manual = probe_rows[VALUE_COL].iloc[6:30].mean()
assert abs(probe_rows["组内滑动均值"].iloc[29] - manual) < 1e-9, \
    f"STATION_C_01 第 29 行窗口值应与手算一致：{probe_rows['组内滑动均值'].iloc[29]:.4f} vs {manual:.4f}"
# 组边界自检：每组首行必为 NaN
for name, sub in curve2.groupby(KEY_COL):
    assert pd.isna(sub["组内滑动均值"].iloc[0]), f"{name} 组首行应为 NaN"
print("综合② 通过 -> 回贴报告：")
for key, value in report.items():
    print(f"   {key:12s} = {value}")
print(f"   对账：STATION_C_01 第 29 行 {probe_rows['组内滑动均值'].iloc[29]:.4f} == 手算 {manual:.4f}")

---

### 复盘提问

**复盘问自己**（六题都能答上来才算过）：

1. `rolling(24).mean()` 在这份数据上为什么会有 319 个 `NaN`？缺失值是怎么"放大"的？
2. 造特征时为什么必须 `shift(1).rolling(24)`？代价是什么？
3. `gb.rolling()` 和 `gb.shift()` 返回的索引差在哪？回贴原表各自要怎么做？
4. `ewm(span=24)`、`ewm(halflife=24)`、`ewm(alpha=0.08)` 是一回事吗？
5. 时间窗口 `rolling("24h")` 的 `min_periods` 默认值是多少？和固定窗口差在哪？
6. 全局阈值和滑动阈值分别回答什么问题？什么业务场景该用哪个？

答不上来的，回 `_notes/错题本.md` 记一笔。